# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [22]:
# Step 0 - Import libraries
# --------------------------

# Import the libraries needed for this section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# For evaluating Regression models (Part 1)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# For evaluating Classification models (Part 2)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For Decision Trees
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree

# For Random Forest
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# For XGBoost
from xgboost import XGBRegressor, XGBClassifier

# Standard seed used throughout the notebook, for reproducibility
SEED = 1

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [4]:
spaceship.shape

(8693, 14)

In [5]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
spaceship_df = spaceship.dropna()

In [7]:
spaceship_df = spaceship_df.drop(columns=["PassengerId", "Name"])

In [8]:
# Select the features (X) and the target (y)

y = spaceship_df['Transported']
X = spaceship_df.drop(columns=["Transported"]).select_dtypes(include=["number"]) # Let's only select numerical columns as our features.

print(X)

       Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck
0     39.0          0.0        0.0           0.0     0.0     0.0
1     24.0        109.0        9.0          25.0   549.0    44.0
2     58.0         43.0     3576.0           0.0  6715.0    49.0
3     33.0          0.0     1283.0         371.0  3329.0   193.0
4     16.0        303.0       70.0         151.0   565.0     2.0
...    ...          ...        ...           ...     ...     ...
8688  41.0          0.0     6819.0           0.0  1643.0    74.0
8689  18.0          0.0        0.0           0.0     0.0     0.0
8690  26.0          0.0        0.0        1872.0     1.0     0.0
8691  32.0          0.0     1049.0           0.0   353.0  3235.0
8692  44.0        126.0     4688.0           0.0     0.0    12.0

[6606 rows x 6 columns]


**Perform Train Test Split**

In [9]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size = 0.2,
    random_state=SEED,
    stratify=y)

# To see the shape of the Train (80%) and Test (20%) datasets
print(f" X_train shape: {X_train.shape}")
print(f" X_test shape: {X_test.shape}")

 X_train shape: (5284, 6)
 X_test shape: (1322, 6)


In [10]:
# Feature Scaling

scaler = StandardScaler()
scaler.set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [16]:
# Decision Tree Classifier
tree_clf = DecisionTreeClassifier(random_state=SEED)
tree_clf.fit(X_train_scaled, y_train)

y_pred_tree = tree_clf.predict(X_test_scaled)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_tree) * 100:.2f}%")

Decision Tree Accuracy: 74.89%


- Bagging and Pasting

- Random Forests

In [23]:
# Random Forest Classifier (Bagging)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_clf.fit(X_train_scaled, y_train)

# Predict and print accuracy
y_pred_rf = rf_clf.predict(X_test_scaled)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf) * 100:.2f}%")

Random Forest Accuracy: 78.14%


In [24]:
# XGBoost Classifier (Boosting)
xgb_clf = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
xgb_clf.fit(X_train_scaled, y_train)

# Predict and print accuracy
y_pred_xgb = xgb_clf.predict(X_test_scaled)
print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%")

XGBoost Accuracy: 78.14%


- Gradient Boosting

In [25]:
# Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
gb_clf.fit(X_train_scaled, y_train)

# Predict and print accuracy
y_pred_gb = gb_clf.predict(X_test_scaled)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb) * 100:.2f}%")

Gradient Boosting Accuracy: 78.29%


- Adaptive Boosting

In [26]:
# AdaBoost Classifier
ada_clf = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
ada_clf.fit(X_train_scaled, y_train)

# Predict and print accuracy
y_pred_ada = ada_clf.predict(X_test_scaled)
print(f"AdaBoost Accuracy: {accuracy_score(y_test, y_pred_ada) * 100:.2f}%")

AdaBoost Accuracy: 75.87%


Which model is the best and why?

In [27]:
# Analysis

# Gradient Boosting scores the highest accuracy with 78.29% and Random Forest and XGBoost tie in second place with 78.14%